In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
path = r"C:\Users\Noble Adike\Desktop\Coding\KPMG\kpmg-1a\data\harvard\Year 2 (Jun 2023 - May 2024)\Local_weather_hourly\48190930_Weather.csv"
df = pd.read_csv(path, encoding="cp1252")

In [3]:
df.head()

,Time,"Air temperature (gund, ¡É)","Relative humidity (gund, %)","Wind speed (gund, m/s)","Weather data, Wind speed (m/s)","Wind direction (gund, ¡Æ)","Weather data, Wind direction","Weather data, Rain","Solar radiation (gund, W/m^2)"
0,2023-06-01 00:00:00-04:00,17.188333,71.258333,0.666667,1.316667,168.916667,280.218572,0,1.000000
1,2023-06-01 01:00:00-04:00,16.398333,75.825000,0.916667,0.934615,147.000000,266.021180,0,1.000000
2,2023-06-01 02:00:00-04:00,15.536667,81.600000,1.208333,1.577778,156.166667,273.063306,0,1.000000
3,2023-06-01 03:00:00-04:00,15.421667,82.825000,0.500000,1.072727,163.833333,274.142811,0,1.000000
4,2023-06-01 04:00:00-04:00,15.007500,84.650000,0.125000,0.608333,174.000000,274.191652,0,1.083333


In [4]:
df.describe()

,"Air temperature (gund, ¡É)","Relative humidity (gund, %)","Wind speed (gund, m/s)","Weather data, Wind speed (m/s)","Wind direction (gund, ¡Æ)","Weather data, Wind direction","Weather data, Rain","Solar radiation (gund, W/m^2)"
count,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000
mean,12.619317,71.923379,1.778254,1.283975,178.794701,204.840103,0.873975,142.455588
std,8.777744,18.237163,1.841943,1.033922,83.813670,103.446760,0.331896,222.067277
min,0.000000,20.966667,0.000000,0.000000,0.000000,0.124811,0.000000,0.620000
25%,4.953750,57.741667,0.458333,0.413826,123.729167,112.987792,1.000000,1.000000
50%,11.847500,73.662500,1.259583,1.110728,182.958333,251.834028,1.000000,5.666667
75%,19.976458,87.993750,2.500000,1.942639,251.083333,291.000000,1.000000,216.854167
max,35.797500,100.000000,16.775000,6.863833,355.000000,359.652274,1.000000,1008.250000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Time                            8784 non-null   object 
 1   Air temperature (gund, ¡É)      8784 non-null   float64
 2   Relative humidity (gund, %)     8784 non-null   float64
 3   Wind speed (gund, m/s)          8784 non-null   float64
 4   Weather data, Wind speed (m/s)  8784 non-null   float64
 5   Wind direction (gund, ¡Æ)       8784 non-null   float64
 6   Weather data, Wind direction    8784 non-null   float64
 7   Weather data, Rain              8784 non-null   int64  
 8   Solar radiation (gund, W/m^2)   8784 non-null   float64
dtypes: float64(7), int64(1), object(1)
memory usage: 617.8+ KB


In [6]:
df['Time'] = pd.to_datetime(df['Time'])
df = df.set_index('Time')

C:\Users\Noble Adike\AppData\Local\Temp\ipykernel_4280\1925714391.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['Time'] = pd.to_datetime(df['Time'])


In [7]:
df = df.rename(columns={
    'Air temperature (gund, ¡É)': 'air_temp_c',
    'Relative humidity (gund, %)': 'rel_humidity_pct',
    'Wind speed (gund, m/s)': 'wind_speed_sensor',
    'Weather data, Wind speed (m/s)': 'wind_speed_weather',
    'Wind direction (gund, ¡Æ)': 'wind_dir_sensor_deg',
    'Weather data, Wind direction': 'wind_dir_weather_deg',
    'Weather data, Rain': 'rain_flag',
    'Solar radiation (gund, W/m^2)': 'solar_ghi_wm2'
})

In [8]:
df['rel_humidity_pct'] = df['rel_humidity_pct'].clip(0, 100)
for c in ['wind_speed_sensor','wind_speed_weather','solar_ghi_wm2']:
    df[c] = df[c].clip(lower=0)

In [9]:
df.isna().sum()

air_temp_c              0
rel_humidity_pct        0
wind_speed_sensor       0
wind_speed_weather      0
wind_dir_sensor_deg     0
wind_dir_weather_deg    0
rain_flag               0
solar_ghi_wm2           0
dtype: int64

In [10]:
good= ['air_temp_c','rel_humidity_pct','wind_speed_sensor','wind_speed_weather','wind_dir_sensor_deg','wind_dir_weather_deg','solar_ghi_wm2']
def treat_outliers_iqr(df):
    df_clean = df.copy()
    for col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1


        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
    return df_clean
df[good] = treat_outliers_iqr(df[good])

In [11]:
df.describe()

,air_temp_c,rel_humidity_pct,wind_speed_sensor,wind_speed_weather,wind_dir_sensor_deg,wind_dir_weather_deg,rain_flag,solar_ghi_wm2
count,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000
mean,12.619317,71.923379,1.686118,1.278394,178.794701,204.840103,0.873975,128.442116
std,8.777744,18.237163,1.529233,1.014842,83.813670,103.446760,0.331896,185.872164
min,0.000000,20.966667,0.000000,0.000000,0.000000,0.124811,0.000000,0.620000
25%,4.953750,57.741667,0.458333,0.413826,123.729167,112.987792,1.000000,1.000000
50%,11.847500,73.662500,1.259583,1.110728,182.958333,251.834028,1.000000,5.666667
75%,19.976458,87.993750,2.500000,1.942639,251.083333,291.000000,1.000000,216.854167
max,35.797500,100.000000,5.562500,4.235859,355.000000,359.652274,1.000000,540.635417


In [12]:
list(df.columns)

['air_temp_c',
 'rel_humidity_pct',
 'wind_speed_sensor',
 'wind_speed_weather',
 'wind_dir_sensor_deg',
 'wind_dir_weather_deg',
 'rain_flag',
 'solar_ghi_wm2']

In [13]:
df.shape

(8784, 8)